[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/03_%E3%82%AB%E3%82%A4%E4%BA%8C%E4%B9%97%E3%81%A8%E5%88%86%E6%95%A3%E5%88%86%E6%9E%90.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計2級-03. カイ二乗検定・分散分析(ANOVA)

**🎯 できるようになること**
- カイ二乗（独立性・適合度）の検定ができる
- 期待度数の意味がわかる
- 一元配置分散分析で3群以上の平均を比較できる

**前提**：統計2級02　/　**所要**：約30分　/　**レベル**：統計検定2級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

## 1. カイ二乗 独立性の検定

「獲得チャネル」と「受注の有無」に関連があるか？
クロス集計表をもとに検定します。

> 📐 **数IIIメモ**：**カイ二乗分布・F分布**も連続分布の一種で、導出は大学範囲。ここでは検定のための“ものさし”として `scipy` に計算してもらい、**p値の読み方**に集中します。

In [ ]:
btob = pd.read_csv('../data/sales_btob.csv')   # 商談データ読み込み
table = pd.crosstab(btob['獲得チャネル'], btob['受注'])  # チャネル×受注のクロス表
print(table)
chi2, p, dof, expected = stats.chi2_contingency(table)  # カイ二乗 独立性の検定
print(f'\nカイ二乗統計量: {chi2:.2f}')
print(f'自由度: {dof},  p値: {p:.5f}')
print('結論:', 'チャネルと受注に関連あり' if p < 0.05 else '関連があるとは言えない')

### 📋 使うデータ：`sales_btob.csv`（架空のBtoB商談400件）
1行＝商談1件。`受注`は 1=成約 / 0=失注。

| 商談ID | 受注日 | 業界 | 獲得チャネル | 商談金額 | 担当者 | 受注 |
|---|---|---|---|---|---|---|
| D0001 | 2026-01-15 | 小売 | 紹介 | 1,211,000 | 佐藤 | 0 |
| D0002 | 2026-02-05 | 医療 | テレアポ | 400,000 | 田中 | 0 |
| D0003 | 2026-02-19 | IT | 紹介 | 542,000 | 高橋 | 1 |

---

### 📋 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

**期待度数**（関連がないと仮定したときの理論値）と実測のズレが大きいほど χ² は大きくなります。

In [ ]:
pd.DataFrame(expected, index=table.index, columns=table.columns).round(1)  # 期待度数（関連が無い場合の理論値）

## 2. カイ二乗 適合度の検定

サイコロを各目こう出た：`[18,21,16,14,17,14]`（計100回）。
「6つの目が均等(各1/6)」といえるか？

In [ ]:
observed = np.array([18, 21, 16, 14, 17, 14])  # 各目の観測回数
expected = np.full(6, observed.sum() / 6)   # 均等なら各回数（期待度数）
chi2, p = stats.chisquare(observed, expected)  # 適合度検定
print(f'chi2={chi2:.2f}, p={p:.3f}')
print('結論:', '均等でない' if p < 0.05 else '均等といえる')

## 3. 一元配置分散分析（ANOVA）

3つの店舗A・B・Cの売上に差があるか？
t検定を3回くり返すと誤りが増えるので、ANOVAで一度に検定します。

In [ ]:
storeA = rng.normal(50, 8, 20)             # 店舗Aの売上
storeB = rng.normal(55, 8, 20)             # 店舗B
storeC = rng.normal(58, 8, 20)             # 店舗C
f_stat, p = stats.f_oneway(storeA, storeB, storeC)  # 一元配置分散分析（3群の平均比較）
print(f'F統計量: {f_stat:.3f},  p値: {p:.4f}')
print('結論:', '少なくとも1店舗は平均が違う' if p < 0.05 else '差は有意でない')

plt.boxplot([storeA, storeB, storeC], tick_labels=['A', 'B', 'C'])  # 箱ひげ図で比較
plt.ylabel('売上'); plt.title('店舗別の売上')
plt.show()

> 📝 ANOVAは「どこかに差がある」までしか分かりません。
「どの店どうしが違うか」は多重比較（Tukey法など）で調べます。

> ⚠️ **よくある間違い**：カイ二乗検定は**度数（カウント）**に使うもの。割合や平均そのものには使いません。また期待度数が小さすぎる（目安5未満）と近似が崩れます。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
from scipy import stats
# Q1: サイコロ観測[18,21,16,14,17,14]の適合度検定のカイ二乗統計量 を ans に
ans = None   # 例: stats.chisquare([18,21,16,14,17,14]).statistic
_check('Q1 カイ二乗', ans, stats.chisquare([18,21,16,14,17,14]).statistic)

In [ ]:
from scipy import stats
# Q2: 3群の分散分析のF統計量 を ans に
ans = None   # 例: stats.f_oneway([50,52,48],[60,61,59],[70,69,71]).statistic
_check('Q2 F統計量', ans, stats.f_oneway([50,52,48],[60,61,59],[70,69,71]).statistic)

---
## 🏆 練習問題

**問1.** `業界` と `受注` に関連があるか、`sales_btob.csv` でカイ二乗検定をしよう。

**問2.** あるアンケートで血液型が `[A,B,O,AB] = [38,22,30,10]`。
日本人の理論比 `[0.38,0.22,0.31,0.09]` に適合するか検定しよう。

**問3.** `students_scores.csv` のクラスA・B・Cで `数学` の平均に差があるか、ANOVAで調べよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3
df = pd.read_csv('../data/students_scores.csv')  # 生徒データを読み込む

> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[03_カイ二乗と分散分析 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/03_%E3%82%AB%E3%82%A4%E4%BA%8C%E4%B9%97%E3%81%A8%E5%88%86%E6%95%A3%E5%88%86%E6%9E%90.md)**

🎉 **統計2級レベル クリア！** ここまでで統計検定2級の主要分野を一通り体験しました。
次は `05_実践_BtoBマーケ` で、学んだ統計を実際のビジネスデータに使ってみよう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 適合度検定 | 理論比に合うか |
| 独立性の検定 | 2つの質的変数に関連があるか |
| 期待度数 | 関連が無い場合の理論値 |
| 一元配置分散分析 | 3群以上の平均の比較 |
| F比 | 群間/群内のばらつきの比 |

In [ ]:
# チートシート（カイ二乗・ANOVA）
from scipy import stats
stats.chi2_contingency([[30,10],[20,40]])   # 独立性の検定
stats.chisquare([18,21,16,14,17,14])        # 適合度検定
stats.f_oneway([1,2,3],[4,5,6],[7,8,9])     # 一元配置分散分析